# FST-Arch: experiments with trained MLPs and ResNets

This notebook runs the procedures of the article on **trained** networks:

* **MLP** (synthetic regression, $d=10$): widths 16 to 1024, depths 1 to 4 (up to about 3M parameters), trained with Adam.
* **ResNet on CIFAR-10** (He et al., 2016): base width $k\in\{8,16,32,64\}$ and $n\in\{1,2,3\}$ blocks per stage (ResNet-8, -14, -20), trained with SGD and data augmentation. Set `BIG = True` for widths up to 128 and ResNet-32.

**Design (pool bootstrap).** Every architecture is trained with the same $M$ seeds, and its per-point losses on a held-out *decision pool* are stored. One Monte Carlo replication resamples $R$ seeds and $n$ points **with replacement** from the pool. These are i.i.d. draws from the pool, so the true risk of each architecture (the pool mean) is known and false rejections can be counted exactly, without training a new network for every replication.

**Runtime.** Use a GPU (Runtime → Change runtime type → GPU). The pools are saved file by file, so an interrupted run resumes where it stopped. Mount Google Drive to keep them across sessions.

In [ ]:
import os, sys, json, time, subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'No GPU found: select a GPU runtime')
if not os.path.exists('fst-architecture-selection'):
    !git clone -q https://github.com/yamenetoo/fst-architecture-selection.git
%cd fst-architecture-selection/experiments/deep

In [ ]:
USE_DRIVE = False   # True: keep the trained pools in Google Drive (recommended for the full run)
POOL_ROOT = '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    POOL_ROOT = '/content/drive/MyDrive/fst_pools'
    os.makedirs(POOL_ROOT, exist_ok=True)
print('pools in', os.path.abspath(POOL_ROOT))

## Settings

`QUICK = True` gives a short run to check that everything works (fewer seeds and epochs). Use `QUICK = False` for the article.

In [ ]:
QUICK = True
BIG = False          # ResNet widths up to 128 and ResNet-32 (n = 5): much longer training

if QUICK:
    MLP_ARGS = '--seeds 6 --epochs 40'
    RES_ARGS = '--widths 16 32 64 --depths 1 2 3 --seeds 6 --epochs 10 --prune-width 64 --prune-depth 3'
    B, B_GRID = 300, 200
else:
    MLP_ARGS = '--seeds 10 --epochs 60'
    RES_ARGS = '--widths 8 16 32 64 --depths 1 2 3 --seeds 10 --epochs 15 --prune-width 64 --prune-depth 3'
    B, B_GRID = 1000, 500
if BIG:
    RES_ARGS = '--widths 16 32 64 128 --depths 1 2 3 5 --seeds 10 --epochs 20 --prune-width 128 --prune-depth 5'

# margins, in units of the loss: squared error (MLP) and 0-1 error (ResNet)
MLP_GAMMA, MLP_DELTA = 0.005, 0.01
RES_GAMMA, RES_DELTA = 0.005, 0.01     # 0.5 and 1 percentage points of test error

## 1. MLP pool (bigger fully connected networks)

In [ ]:
t0 = time.time()
!python train_pool.py --task mlp {MLP_ARGS} --out {POOL_ROOT}/pool_mlp
print(f'MLP pool: {(time.time() - t0) / 60:.1f} min')

The analysis first prints the pool risks and which null hypotheses are true for the chosen $\gamma$. If every null is false (or every null is true), the FWER column carries no information: change `MLP_GAMMA` and run the analysis again (no retraining is needed).

In [ ]:
!python analyze_pool.py --task mlp --pool {POOL_ROOT}/pool_mlp --gamma {MLP_GAMMA} --delta {MLP_DELTA} --B {B} --B-grid {B_GRID}

## 2. ResNet pool on CIFAR-10

On **Kaggle with two GPUs**, replace the training cell by:
```
!python train_pool.py --task resnet {RES_ARGS} --out {POOL_ROOT}/pool_resnet --device cuda:0 --shard 0/2 & \
 python train_pool.py --task resnet {RES_ARGS} --out {POOL_ROOT}/pool_resnet --device cuda:1 --shard 1/2 ; wait
```

In [ ]:
t0 = time.time()
!python train_pool.py --task resnet {RES_ARGS} --out {POOL_ROOT}/pool_resnet
print(f'ResNet pool: {(time.time() - t0) / 60:.1f} min')

In [ ]:
!python analyze_pool.py --task resnet --loss err --pool {POOL_ROOT}/pool_resnet --gamma {RES_GAMMA} --delta {RES_DELTA} --B {B} --B-grid {B_GRID}

## 3. Plots

Left: exact size at the boundary against the decision-sample size $n$ (nominal 0.05; a calibrated test stays on the line). Middle: FWER of the growing procedure. Right: pruning, probability that the returned network loses at least $\delta$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
methods = ['crossed', 'seed_only', 'single', 'welch']
colors = ['#4c72b0', '#dd8452', '#c44e52', '#8172b3']
for tag in ['mlp_se', 'resnet_err']:
    f = f'results_deep_{tag}.json'
    if not os.path.exists(f):
        continue
    r = json.load(open(f))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    keys = list(r['size'])
    ns = sorted({int(k.split(',')[0][2:]) for k in keys}); Rs = sorted({int(k.split(',')[1][2:]) for k in keys})
    for m, c in zip(methods, colors):
        for R, ls in zip(Rs, [':', '-', '--', '-.']):
            ax[0].plot(ns, [r['size'][f'n={n},R={R}'][m]['mean'] for n in ns], ls=ls, color=c, marker='o',
                       label=m if R == Rs[len(Rs) // 2] else None)
    ax[0].axhline(r['alpha'], color='k', lw=1); ax[0].set_xscale('log'); ax[0].set_xlabel('n')
    ax[0].set_title(f'{tag}: exact size (line style = R in {Rs})'); ax[0].legend(fontsize=8)
    g = r['grow']; names = list(g)
    ax[1].bar(names, [g[m]['fwer'] for m in names], yerr=[1.96 * g[m]['fwer_se'] for m in names], color='#4c72b0')
    ax[1].axhline(r['alpha'], ls='--', color='k', lw=1); ax[1].set_title('growing: FWER'); ax[1].tick_params(axis='x', rotation=30)
    if 'prune' in r:
        p = r['prune']
        ax[2].bar(['cumulative', 'stepwise'], [p[k]['harm_exceeds'] for k in ['cumulative', 'stepwise']],
                  yerr=[1.96 * p[k]['harm_exceeds_se'] for k in ['cumulative', 'stepwise']], color=['#4c72b0', '#c44e52'])
        ax[2].axhline(r['alpha'], ls='--', color='k', lw=1)
        ax[2].set_title(r'pruning: $P\{\mu(\hat a)-\mu(a_0)\geq\delta\}$')
    plt.tight_layout(); plt.savefig(f'deep_{tag}.png', dpi=200); plt.show()

In [ ]:
files_out = [f for f in os.listdir('.') if f.startswith(('results_deep_', 'table_deep_', 'deep_')) and not f.endswith('.py')]
print(files_out)
try:
    from google.colab import files
    for f in files_out:
        files.download(f)
except ImportError:
    pass